# Inspect videos, imagery, and the PDF report

The pipeline produces several visualisations that are easier to grok in
a browser than to describe in code:

- True-colour Sentinel-2 timeline (MP4)
- True-colour + paddock-boundary overlay (MP4)
- Fractional-cover false-colour timeline (MP4, R=bare / G=green / B=non-green)
- Fractional-cover + paddock-boundary overlay (MP4)
- Per-paddock thumbnail calendar (PNG per year × page chunk)
- Stitched A4-landscape PDF report combining everything

This notebook shows how to embed each of those inline in Jupyter so you
can review without leaving the kernel.

> **Prerequisite.** A pipeline run for the query below.


In [ ]:
from datetime import date
from pathlib import Path
from PaddockTS.query import Query

query = Query(
    bbox=[148.36265, -33.52606, 148.38265, -33.50606],
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),
    stub="stages_demo",
)
out = Path(query.out_dir)
print("outputs:", out)


## 1. What did the pipeline produce?


In [ ]:
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:60s}  {f.stat().st_size / 1e6:>6.2f} MB")


## 2. True-colour Sentinel-2 timeline

Each frame is a normalised RGB composite from the visible bands, date
stamped in the corner.


In [ ]:
from IPython.display import Video

mp4 = next(out.glob("*_sentinel2.mp4"), None)
print(mp4)
Video(str(mp4), embed=True, html_attributes="controls loop") if mp4 else None


## 3. Sentinel-2 with paddock boundaries overlaid

Red polygon outlines, paddock IDs at each polygon's centroid. Useful
for eyeballing segmentation quality against the underlying imagery.


In [ ]:
mp4 = next(out.glob("*_sentinel2_paddocks.mp4"), None)
print(mp4 or "(no paddocks-overlay video found — has the SAM stage finished?)")
Video(str(mp4), embed=True, html_attributes="controls loop") if mp4 else None


## 4. Fractional cover (false-colour)

R = bare ground, G = green vegetation, B = non-green vegetation. Fully
red = bare, fully green = growing, fully blue = stubble / dry.


In [ ]:
mp4 = next(out.glob("*_fractional_cover.mp4"), None)
print(mp4)
Video(str(mp4), embed=True, html_attributes="controls loop") if mp4 else None


## 5. Per-paddock thumbnail calendar

One PNG per year (split across pages if there are more paddocks than
`max_paddocks_per_page`). Each row is one paddock; columns are 48
evenly-spaced slots across the year. Useful for spotting cloud
problems, cropping events, or stand-out paddocks at a glance.


In [ ]:
from IPython.display import Image as NBImage, display

for png in sorted(out.glob("*_calendar_*.png")):
    print(png.name)
    display(NBImage(filename=str(png)))


### Notes on inline video embedding

`Video(str(path), embed=True, ...)` reads the MP4 bytes and inlines
them into the notebook output cell. The `.ipynb` file grows by the
size of the embedded video — fine for a few MB clips, less great for
multi-year runs. Pass `embed=False` to reference the file path
instead; only works while the file lives at that path (so the notebook
stops rendering the video if you move / delete the run output).


## 6. The stitched PDF report

The full report combining topography, climate, calendar, and
phenology is generated as `{query.stub}_report.pdf` under
`query.out_dir`. See [`08_inspect_pdf.ipynb`](08_inspect_pdf.ipynb)
for an inline preview.


## See also

- [`PaddockTS.Plotting.sentinel2_video`](https://thestochasticman.github.io/paddock-ts-local/api/plotting/#sentinel-2-videos)
- [`PaddockTS.Plotting.calendar_plot`](https://thestochasticman.github.io/paddock-ts-local/api/plotting/#calendar-plot)
- [`PaddockTS.Plotting.make_pdf`](https://thestochasticman.github.io/paddock-ts-local/api/plotting/#pdf-report)
